In [0]:
# Create a volume in the workspace for storing IoT sensor data if it does not already exist
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.iot_sensor_data
COMMENT 'Volume for IoT sensor data'
""")

In [0]:
print('Dataset')

In [0]:
import json
import os
import time
import random
import threading
from datetime import datetime
import numpy as np
import uuid

# =========================================================
# CONFIG
# =========================================================

READINGS_PER_MACHINE = 100  # Number of readings to generate per machine

# 1 minute interval (in seconds)
SLEEP_TIME = 30

OUTPUT_PATH = "/Volumes/workspace/default/iot_sensor_data"  # Output path for sensor data

MACHINES = [
    "M_1",
    "M_2",
    "M_3"
]

MACHINE_TYPES = [
    "Motor",
    "CNC",
    "Transformer"
]

# =========================================================
# BASE SENSOR PROFILES
# =========================================================

# Baseline sensor values for each machine type
BASE = {

    "Motor": {
        "temp": 65,
        "vibration": 3,
        "current": 20,
        "voltage": 415,
        "pressure": 5,
        "rpm": 1800
    },

    "CNC": {
        "temp": 55,
        "vibration": 2,
        "current": 15,
        "voltage": 380,
        "pressure": 4,
        "rpm": 1200
    },

    "Transformer": {
        "temp": 60,
        "vibration": 1.5,
        "current": 30,
        "voltage": 11000,
        "pressure": 6,
        "rpm": 0
    }
}

# =========================================================
# MACHINE RUNTIME STORAGE
# =========================================================

machines_runtime = {}  # Stores runtime and counters for each machine

# =========================================================
# POWER CALCULATION
# =========================================================

def calculate_power(
    voltage,
    current,
    power_factor=0.85
):
    # Calculate 3-phase power in kW
    power = (
        np.sqrt(3)
        * voltage
        * current
        * power_factor
    ) / 1000

    return float(round(power, 2))

# =========================================================
# ACOUSTIC FEATURES
# =========================================================

def generate_acoustic_features(vibration):
    # Generate synthetic acoustic emission features based on vibration
    ae_rms = round(
        vibration * random.uniform(0.8, 1.5),
        3
    )

    ae_peak = round(
        ae_rms * random.uniform(1.5, 3.0),
        3
    )

    ae_energy = round(
        (ae_rms ** 2)
        * random.uniform(10, 30),
        3
    )

    return {

        "ae_rms": ae_rms,
        "ae_peak": ae_peak,
        "ae_energy": ae_energy
    }

# =========================================================
# INTERNAL MACHINE CONDITION
# NOT STORED IN OUTPUT
# =========================================================

def determine_internal_condition():
    # Randomly determine the internal condition/state of the machine
    probability = random.random()

    # RUNNING
    if probability < 0.70:
        return "RUNNING"

    # IDLE
    elif probability < 0.82:
        return "IDLE"

    # STOPPED
    elif probability < 0.90:
        return "STOPPED"

    # MAINTENANCE
    elif probability < 0.95:
        return "MAINTENANCE"

    # FAULT
    else:
        return "FAULT"

# =========================================================
# SENSOR GENERATION
# =========================================================

def generate_sensor_values(
    base,
    machine_type,
    internal_condition
):
    # Generate sensor values based on machine type and condition
    voltage = round(
        base["voltage"]
        + np.random.normal(0, 3),
        2
    )
    # =====================================================
    # RUNNING
    # =====================================================
    
    if internal_condition == "RUNNING":
        print('RUNNING')
        temperature = round(
            base["temp"]
            + np.random.normal(0, 2),
            2
        )

        vibration = round(
            base["vibration"]
            + np.random.normal(0, 0.3),
            2
        )

        current = round(
            base["current"]
            + np.random.normal(0, 2),
            2
        )

        pressure = round(
            base["pressure"]
            + np.random.normal(0, 0.5),
            2
        )

        if machine_type == "Transformer":

            rpm = 0

        else:

            rpm = round(
                base["rpm"]
                + np.random.normal(0, 50),
                2
            )
    
    # =====================================================
    # IDLE
    # =====================================================

    elif internal_condition == "IDLE":
        print('IDLE')
        temperature = round(
            random.uniform(35, 50),
            2
        )

        vibration = round(
            random.uniform(0.5, 1.5),
            2
        )

        current = round(
            random.uniform(3, 8),
            2
        )

        pressure = round(
            random.uniform(1, 3),
            2
        )

        if machine_type == "Transformer":

            rpm = 0

        else:

            rpm = round(
                random.uniform(50, 200),
                2
            )
    
    # =====================================================
    # STOPPED
    # =====================================================

    elif internal_condition == "STOPPED":
        print('STOPPED')
        temperature = round(
            random.uniform(25, 40),
            2
        )

        vibration = round(
            random.uniform(0.1, 0.5),
            2
        )

        current = round(
            random.uniform(0, 2),
            2
        )

        pressure = round(
            random.uniform(0, 1),
            2
        )

        rpm = 0
    
    # =====================================================
    # MAINTENANCE
    # =====================================================

    elif internal_condition == "MAINTENANCE":
        print('MAINTENANCE')
        temperature = round(
            random.uniform(20, 35),
            2
        )

        vibration = round(
            random.uniform(0, 0.2),
            2
        )

        current = round(
            random.uniform(0, 1),
            2
        )

        pressure = round(
            random.uniform(0, 0.5),
            2
        )

        rpm = 0
    
    # =====================================================
    # FAULT
    # =====================================================

    else:
        print('ELSE')
        temperature = round(
            random.uniform(90, 130),
            2
        )

        vibration = round(
            random.uniform(8, 15),
            2
        )

        current = round(
            random.uniform(
                base["current"] * 1.5,
                base["current"] * 2.2
            ),
            2
        )

        pressure = round(
            random.uniform(
                base["pressure"] * 1.5,
                base["pressure"] * 2
            ),
            2
        )

        if machine_type == "Transformer":

            rpm = 0

        else:

            rpm = round(
                random.uniform(
                    base["rpm"] * 0.5,
                    base["rpm"] * 1.5
                ),
                2
            )
    
    power_kw = calculate_power(
        voltage,
        current
    )
    
    ae_features = generate_acoustic_features(
        vibration
    )
    
    return {

        "temperature": temperature,
        "vibration": vibration,
        "current": current,
        "voltage": voltage,
        "pressure": pressure,
        "rpm": rpm,
        "power_kw": power_kw,

        **ae_features
    }

# =========================================================
# UPDATE OPERATIONAL COUNTERS
# =========================================================

def update_operational_counters(
    machine_id,
    sensors,
    machine_type,
    internal_condition
):
    # Update runtime, cycle, alarm, start, and stop counters for the machine
    runtime_increment = (
        SLEEP_TIME / 3600
    )
    # =====================================================
    # RUNTIME
    # =====================================================
    if internal_condition in [
        "RUNNING",
        "IDLE"
    ]:

        machines_runtime[machine_id][
            "runtime_hours"
        ] += runtime_increment

    # =====================================================
    # CYCLE COUNT
    # =====================================================

    cycle_increment = 0

    if internal_condition == "RUNNING":

        rpm = sensors["rpm"]

        if machine_type == "Motor":

            cycle_increment = max(
                1,
                int(rpm / 300)
            )

        elif machine_type == "CNC":

            cycle_increment = max(
                1,
                int(rpm / 250)
            )

    machines_runtime[machine_id][
        "cycle_count"
    ] += cycle_increment

    # =====================================================
    # ALARM COUNT
    # =====================================================

    if internal_condition == "FAULT":

        machines_runtime[machine_id][
            "alarm_count"
        ] += random.randint(1, 3)

    # =====================================================
    # START / STOP EVENTS
    # =====================================================

    if internal_condition == "RUNNING":

        if random.random() < 0.005:

            machines_runtime[machine_id][
                "start_count"
            ] += 1

    elif internal_condition in [
        "STOPPED",
        "MAINTENANCE"
    ]:

        if random.random() < 0.005:

            machines_runtime[machine_id][
                "stop_count"
            ] += 1

    return {

        "runtime_hours": round(
            machines_runtime[machine_id][
                "runtime_hours"
            ],
            3
        ),

        "cycle_count":
            machines_runtime[machine_id][
                "cycle_count"
            ],

        "alarm_count":
            machines_runtime[machine_id][
                "alarm_count"
            ],

        "start_count":
            machines_runtime[machine_id][
                "start_count"
            ],

        "stop_count":
            machines_runtime[machine_id][
                "stop_count"
            ]
    }

# =========================================================
# FILE PATH
# =========================================================

def get_output_file(machine_id):
    # Generate output file path for the machine's data
    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    dbutils.fs.mkdirs(f"{OUTPUT_PATH}")

    return (
        f"""{OUTPUT_PATH}/{datetime.now().strftime('%Y')}/{datetime.now().strftime('%m')}/{datetime.now().strftime('%d')}/{machine_id}_{timestamp}.json"""
    )

# =========================================================
# INITIALIZE MACHINES
# =========================================================

def initialize_machine(
    machine_id,
    machine_type
):
    # Initialize counters and runtime for a machine
    if machine_type == "Motor":

        cycle_count = random.randint(
            500000,
            3000000
        )

        runtime_hours = random.uniform(
            12000,
            40000
        )

    elif machine_type == "CNC":

        cycle_count = random.randint(
            100000,
            800000
        )

        runtime_hours = random.uniform(
            8000,
            25000
        )

    else:

        cycle_count = 0

        runtime_hours = random.uniform(
            30000,
            100000
        )

    machines_runtime[machine_id] = {

        "runtime_hours":
            runtime_hours,

        "cycle_count":
            cycle_count,

        "alarm_count":
            random.randint(10, 200),

        "start_count":
            random.randint(100, 5000),

        "stop_count":
            random.randint(100, 5000)
    }

# =========================================================
# MACHINE THREAD
# =========================================================

def run_machine(
    machine_id,
    machine_type
):  
    # Simulate a machine generating sensor data and writing to file
    initialize_machine(
        machine_id,
        machine_type
    )
    file_path = get_output_file(
        machine_id
    )
    print(file_path)
    all_records = []
    base = BASE[machine_type]
    print(
        f"\nStarting Machine: "
        f"{machine_id}"
    )
    for reading_num in range(
        READINGS_PER_MACHINE
    ):
        internal_condition = (
            determine_internal_condition()
        )
        sensors = generate_sensor_values(
            base,
            machine_type,
            internal_condition
        )
        counters = (
            update_operational_counters(
                machine_id,
                sensors,
                machine_type,
                internal_condition
            )
        )
        # =================================================
        # FINAL RECORD
        # NO MACHINE STATE STORED
        # =================================================

        record = {
            "sensor_event_id":
                str(uuid.uuid4()),

            "publish_timestamp":
                datetime.utcnow().isoformat(),

            "machine_id":
                machine_id,

            "machine_type":
                machine_type,

            **sensors,

            **counters
        }

        print(record)
        time.sleep(SLEEP_TIME)
        all_records.append(record)
        
    json_strings = '\n'.join(json.dumps(item) for item in all_records)
    dbutils.fs.put(file_path, json_strings + "\n", True)
    print('writing')

    print(
        f"\nCompleted Machine: "
        f"{machine_id}"
    )
# =========================================================
# MAIN
# =========================================================

def run():
    # Start threads for each machine simulation
    threads = []

    for i, machine_id in enumerate(MACHINES):
        machine_type = MACHINE_TYPES[
            i % len(MACHINE_TYPES)
        ]
        thread = threading.Thread(
            target=run_machine,
            args=(
                machine_id,
                machine_type
            )
        )
        thread.start()
        threads.append(thread)

    for thread in threads:

        thread.join()

    print("\n================================")
    print("ALL MACHINES COMPLETED")
    print("================================")

# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    run()